# osmFISH
This portable notebook uses the released MAFWT package and SC_BC_pipeline. Install R and mclust before importing MAFWT. The single section uses slice_id=0 and zero cross-section neighbors.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "Reproduction_Notebook":
    ROOT = ROOT.parent
if not (ROOT / "MAFWT" / "Pipeline.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root or Reproduction_Notebook")
sys.path.insert(0, str(ROOT))


In [ ]:
import warnings

from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
import torch

import pandas as pd
import numpy as np
import scanpy as sc
import os
import yaml
from pathlib import Path

from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.decomposition import PCA



# %%
import sys
import MAFWT as TOOLS

def load_data(config):
    adata = sc.read(ROOT / 'datasets' / 'osmFISH' / 'adata.h5ad')
    adata.var_names_make_unique()
    adata.obs['slice_id'] = 0

    ##### Load layer_guess label, if have
    adata.obs['layer_guess'] = adata.obs['cluster']

    edge_index = TOOLS.graph_construction(adata, config['data']['k_cutoff'])

    adata.layers['count'] = adata.X
    sc.pp.filter_genes(adata, min_cells=50)
    sc.pp.filter_genes(adata, min_counts=10)
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer='count', n_top_genes=config['data']['top_genes'])
    adata = adata[:, adata.var['highly_variable'] == True]
    sc.pp.scale(adata)

    adata.obsm['X_pca'] = adata.X
    return adata, edge_index


In [ ]:
with open(ROOT / 'Config' / 'osmFISH.yaml', 'r', encoding='utf-8') as f:
    config = yaml.load(f.read(), Loader=yaml.FullLoader)
config['train']['topk_neighs_inter'] = 0
adata, edge_index = load_data(config)


In [ ]:
adata, edge_index = load_data(config)


In [ ]:
import copy
import random
import sys

num_clusters = 11
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
clusType = "mclust"

results = []
seeds = [3407, 42, 123, 2023, 777]

for seed in seeds:
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed_all(0)

    cfg = copy.deepcopy(config)
    cfg['seed'] = seed

    print(f"\n=== seed={seed} ===", flush=True)
    net = TOOLS.SC_BC_pipeline(adata, edge_index=edge_index, num_clusters=num_clusters,
                            device=device, config=cfg, imputation=False)
    net.trian()
    enc_rep, recon = net.process()

    adata.obs[clusType] = TOOLS.clustering(z=enc_rep, n_clust=num_clusters, num_seed=1, method=clusType)
    adata.obs[clusType] = TOOLS.refine_label(adata, 50, key=clusType)
    ARI, ACC, DIS = TOOLS.get_metrics(adata, 'layer_guess', clusType)
    results.append({'seed': seed, 'ARI': ARI, 'ACC': ACC})
    print(f"ARI={ARI:.4f} ACC={ACC:.4f}", flush=True)

df = pd.DataFrame(results)
print(f"\n===== Five runs =====", flush=True)
print(df.to_string(index=False))
print(f"ARI: mean={df['ARI'].mean():.4f}  std={df['ARI'].std():.4f}  range=[{df['ARI'].min():.4f}, {df['ARI'].max():.4f}]")
print(f"ACC: mean={df['ACC'].mean():.4f}  std={df['ACC'].std():.4f}")


In [ ]:


import warnings
warnings.filterwarnings("ignore")
import torch
import numpy as np
import scanpy as sc
import os
import yaml


import sys
import MAFWT as TOOLS


def load_data(config):
    adata = sc.read(ROOT / 'datasets' / 'osmFISH' / 'adata.h5ad')
    adata.var_names_make_unique()
    adata.obs['slice_id'] = 0
    adata.obs['layer_guess'] = adata.obs['cluster']
    edge_index = TOOLS.graph_construction(adata, config['data']['k_cutoff'])
    adata.layers['count'] = adata.X
    sc.pp.filter_genes(adata, min_cells=50)
    sc.pp.filter_genes(adata, min_counts=10)
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer='count', n_top_genes=config['data']['top_genes'])
    adata = adata[:, adata.var['highly_variable'] == True]
    sc.pp.scale(adata)
    adata.obsm['X_pca'] = adata.X
    return adata, edge_index


# Fix random seed
import random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

# Load config & data
with open(ROOT / 'Config' / 'osmFISH.yaml', 'r', encoding='utf-8') as f:
    config = yaml.load(f.read(), Loader=yaml.FullLoader)
config['train']['topk_neighs_inter'] = 0
config['seed'] = 3407

adata, edge_index = load_data(config)

# Train
num_clusters = 11
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = TOOLS.SC_BC_pipeline(adata, edge_index=edge_index, num_clusters=num_clusters,
                        device=device, config=config, imputation=False)
net.trian()

# Inference + clustering
enc_rep, recon = net.process()
adata.obsm['latent'] = enc_rep
adata.obsm['recon'] = recon

clusType = "mclust"
adata.obs[clusType] = TOOLS.clustering(z=enc_rep, n_clust=num_clusters, num_seed=1, method=clusType)
adata.obs[clusType] = TOOLS.refine_label(adata, 50, key=clusType)

ARI, ACC, DIS = TOOLS.get_metrics(adata, 'layer_guess', clusType)
print(f"ARI: {round(ARI, 4)} ACC: {round(ACC, 4)}")
print(f"Number of clusters: {len(set(adata.obs[clusType]))}")

# ===== Fig 1: Spatial clustering =====
sc.pl.spatial(adata, color=[clusType],
              title=f'ARI:{round(ARI, 3)} ACC:{round(ACC, 3)}',
              spot_size=500)

# ===== Fig 2: Ground truth vs prediction spatial =====
sc.pl.spatial(adata, color=['layer_guess', clusType],
              title=[f'Ground Truth', f'MAFWT (ARI={round(ARI,3)})'],
              spot_size=500)

# ===== Fig 3: UMAP visualization =====
sc.pp.neighbors(adata, use_rep='latent', n_neighbors=15)
sc.tl.umap(adata, min_dist=0.3)
sc.pl.umap(adata, color=['layer_guess', clusType],
           title=[f'Ground Truth', f'MAFWT (ARI={round(ARI,3)})'])
